# AR Dunning Multi-Agent System

LangChain multi-agent pipeline for Accounts Receivable dunning, built with **Claude Agent Teams**.

## Claude Agent Teams used to build this project (development only — not runtime agents)
| Agent | Responsibility |
|---|---|
| **Tools Agent** | `send_invoice_to_customer`, `update_inquiry_status_database`, `update_erp` |
| **Agent Creation Agent** | `ParseCustomerResponse` and `AccessUpdateERP` LangChain agents |
| **Orchestrator Agent** | `orchestrate()`, `generate_status_report()`, `reset_database()` |

## Processing Flow
```
Input_Emails/*.txt
  └─► ParseCustomerResponse  (gemini-3.1-flash-lite)
        ├─ invoice_request  →  send_invoice_to_customer  →  Output/
        └─ promise_date     →  AccessUpdateERP (gemini-3.1-flash-lite)  →  ERPDirectory/invoices.xlsx
        └─ (both paths)     →  update_inquiry_status_database  →  invoice_inquiry.db
  └─► Move email to Processed_Emails/

After all emails → StatusReport saved to Output/
```

## LLM Agents & Model
There are **two LangChain LLM agents** in this system, both using `gemini-3.1-flash-lite`:

| LLM Agent | Purpose | Model |
|---|---|---|
| `ParseCustomerResponse` | Classifies customer emails; calls send-invoice or DB-update tools | `gemini-3.1-flash-lite` |
| `AccessUpdateERP` | Updates ERP Excel with customer promise dates | `gemini-3.1-flash-lite` |

> `orchestrate()` and `generate_status_report()` are plain Python functions — they are **not** LLM agents.

In [92]:
# Uncomment and run once to install dependencies
# !pip install langchain langchain-google-genai google-generativeai pandas openpyxl fpdf2

In [93]:
import time
import os
import json
import shutil
import sqlite3
import traceback
from datetime import datetime
from pathlib import Path

import pandas as pd
from fpdf import FPDF
from pydantic import BaseModel, Field
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.prebuilt import create_react_agent

In [94]:
import os
from pathlib import Path as _P
_env = _P('.env')
if _env.exists():
    for _ln in _env.read_text().splitlines():
        _ln = _ln.strip()
        if _ln and not _ln.startswith('#') and '=' in _ln:
            _k, _, _v = _ln.partition('=')
            os.environ.setdefault(_k.strip(), _v.strip().strip("'\""))

GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY', '')
if not GOOGLE_API_KEY:
    raise ValueError('GOOGLE_API_KEY not set. Add it to .env or set the env var.')

# Two LLM agents, both using the same model.
# 'orchestrator' and 'report' are plain Python — no LLM agent needed for them.
MODELS = {
    'parse_agent': 'gemini-3.1-flash-lite',  # ParseCustomerResponse LLM agent
    'erp_agent':   'gemini-3.1-flash-lite',  # AccessUpdateERP LLM agent
}

BASE_DIR             = Path('.')
INPUT_EMAILS_DIR     = BASE_DIR / 'Input_Emails'
PROCESSED_EMAILS_DIR = BASE_DIR / 'Processed_Emails'
INPUT_INVOICES_DIR   = BASE_DIR / 'Input' / 'Invoices'
OUTPUT_DIR           = BASE_DIR / 'Output'
ERP_DIR              = BASE_DIR / 'ERPDirectory'
DB_PATH              = BASE_DIR / 'invoice_inquiry.db'
ERP_EXCEL_PATH       = ERP_DIR / 'invoices.xlsx'

In [95]:
def setup_directories():
    for d in [INPUT_EMAILS_DIR, PROCESSED_EMAILS_DIR, INPUT_INVOICES_DIR, OUTPUT_DIR, ERP_DIR]:
        d.mkdir(parents=True, exist_ok=True)
    print("Directories ready.")


def setup_database():
    conn = sqlite3.connect(DB_PATH)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS invoice_inquiry (
            id             INTEGER PRIMARY KEY AUTOINCREMENT,
            invoice_number TEXT NOT NULL,
            customer_name  TEXT,
            customer_email TEXT,
            inquiry_type   TEXT NOT NULL,
            promise_date   TEXT,
            status         TEXT DEFAULT 'processed',
            email_file     TEXT,
            notes          TEXT,
            processed_at   TEXT NOT NULL
        )
    """)
    conn.commit()
    conn.close()
    print("Database ready.")


setup_directories()
setup_database()

Directories ready.
Database ready.


In [96]:
# ── Invoice master data (shared by PDF creator and ERP seeder) ──
INVOICE_DATA = [
    {"number": "INV-001", "customer": "ACME Corp",         "email": "john.smith@acmecorp.com",       "amount": 5000.00,  "date": "2026-03-01", "due": "2026-04-01"},
    {"number": "INV-002", "customer": "Tech Startup Inc",  "email": "sarah.jones@techstartup.io",    "amount": 12500.00, "date": "2026-03-15", "due": "2026-04-15"},
    {"number": "INV-003", "customer": "Global Retail Ltd", "email": "mike.chen@globalretail.com",    "amount": 3250.00,  "date": "2026-04-01", "due": "2026-05-01"},
    {"number": "INV-004", "customer": "Manufacturing Net", "email": "emily.white@manufacturing.net", "amount": 8750.00,  "date": "2026-03-20", "due": "2026-04-20"},
    {"number": "INV-005", "customer": "Logistics Co",      "email": "david.brown@logistics.co",      "amount": 15200.00, "date": "2026-04-10", "due": "2026-05-10"},
]


def create_sample_invoices():
    """Create simple PDF invoices in Input/Invoices if they do not already exist."""
    for inv in INVOICE_DATA:
        pdf_path = INPUT_INVOICES_DIR / f"{inv['number']}.pdf"
        if pdf_path.exists():
            continue
        pdf = FPDF()
        pdf.add_page()
        pdf.set_font("Helvetica", "B", 20)
        pdf.cell(0, 15, "INVOICE", ln=True, align="C")
        pdf.line(10, pdf.get_y(), 200, pdf.get_y())
        pdf.ln(5)
        pdf.set_font("Helvetica", "", 11)
        for label, value in [
            ("Invoice Number", inv["number"]),
            ("Customer",       inv["customer"]),
            ("Customer Email", inv["email"]),
            ("Amount Due",     f"${inv['amount']:,.2f}"),
            ("Invoice Date",   inv["date"]),
            ("Due Date",       inv["due"]),
        ]:
            pdf.cell(60, 8, f"{label}:", ln=False)
            pdf.cell(0,  8, value, ln=True)
        pdf.ln(10)
        pdf.set_font("Helvetica", "I", 10)
        pdf.cell(0, 8, "Thank you for your business.", ln=True, align="C")
        pdf.output(str(pdf_path))
    print(f"Sample invoices ready in {INPUT_INVOICES_DIR}")


create_sample_invoices()

Sample invoices ready in Input\Invoices


In [97]:
def create_erp_excel():
    """Seed ERPDirectory/invoices.xlsx with invoice data if it does not exist."""
    if ERP_EXCEL_PATH.exists():
        print(f"ERP file already exists: {ERP_EXCEL_PATH}")
        return
    df = pd.DataFrame([{
        "Invoice Number": inv["number"],
        "Customer Name":  inv["customer"],
        "Customer Email": inv["email"],
        "Invoice Amount": inv["amount"],
        "Invoice Date":   inv["date"],
        "Due Date":       inv["due"],
        "Status":         "Overdue",
        "Promise Date":   "",
        "Notes":          "",
    } for inv in INVOICE_DATA])
    df.to_excel(ERP_EXCEL_PATH, index=False)
    print(f"ERP Excel created: {ERP_EXCEL_PATH}")


create_erp_excel()

ERP file already exists: ERPDirectory\invoices.xlsx


In [98]:
# === TOOLS  (written by the Tools Agent during project build) ===


@tool
def send_invoice_to_customer(invoice_number: str) -> str:
    """
    Sends a copy of an invoice PDF to the customer by copying it from the input invoices
    directory to the output directory. Use this tool when a customer has requested a copy
    of their invoice. The invoice_number should match the PDF filename (without extension).
    """
    source_path      = INPUT_INVOICES_DIR / f'{invoice_number}.pdf'
    destination_path = OUTPUT_DIR / f'{invoice_number}.pdf'
    if not source_path.exists():
        return f"Error: Invoice PDF for '{invoice_number}' not found at '{source_path}'."
    try:
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        shutil.copy2(str(source_path), str(destination_path))
        ts = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        return f"[{ts}] Invoice '{invoice_number}' copied to Output/."
    except Exception as e:
        return f"Error sending invoice '{invoice_number}': {str(e)}"


@tool
def update_inquiry_status_database(
    invoice_number: str,
    customer_name: str,
    customer_email: str,
    inquiry_type: str,
    promise_date: str = '',
    notes: str = '',
) -> str:
    """
    Records a processed customer inquiry into the SQLite database. Use this tool after
    handling any customer email to log the outcome. The inquiry_type must be either
    'invoice_request' (customer asked for a copy of their invoice) or 'promise_date'
    (customer provided a date by which they promise to pay). Optionally supply a
    promise_date (YYYY-MM-DD) and any notes about the interaction.
    """
    if inquiry_type not in ('invoice_request', 'promise_date'):
        return f"Error: inquiry_type must be 'invoice_request' or 'promise_date', got '{inquiry_type}'."
    try:
        conn   = sqlite3.connect(str(DB_PATH))
        cursor = conn.cursor()
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS invoice_inquiry (
                id             INTEGER PRIMARY KEY AUTOINCREMENT,
                invoice_number TEXT NOT NULL,
                customer_name  TEXT NOT NULL,
                customer_email TEXT NOT NULL,
                inquiry_type   TEXT NOT NULL,
                promise_date   TEXT,
                status         TEXT DEFAULT 'processed',
                email_file     TEXT,
                notes          TEXT,
                processed_at   TEXT NOT NULL
            )
        """)
        processed_at = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        cursor.execute(
            """
            INSERT INTO invoice_inquiry
                (invoice_number, customer_name, customer_email, inquiry_type,
                 promise_date, status, email_file, notes, processed_at)
            VALUES (?, ?, ?, ?, ?, 'processed', NULL, ?, ?)
            """,
            (
                invoice_number, customer_name, customer_email, inquiry_type,
                promise_date if promise_date else None,
                notes        if notes        else None,
                processed_at,
            ),
        )
        conn.commit()
        conn.close()
        ts = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        return (
            f"[{ts}] DB updated: invoice='{invoice_number}' customer='{customer_name}' "
            f"type='{inquiry_type}' promise='{promise_date}'."
        )
    except Exception as e:
        return f"Error updating DB for invoice '{invoice_number}': {str(e)}"


@tool
def update_erp(
    invoice_number: str,
    promise_date: str,
    customer_name: str = '',
    notes: str = '',
) -> str:
    """
    Updates the ERP Excel file with a customer payment promise date. Use this tool
    when a customer has provided a date by which they promise to pay. Finds the invoice
    row by invoice_number, sets Promise Date to promise_date (YYYY-MM-DD format),
    and changes Status to 'Promise Received'.
    """
    if not ERP_EXCEL_PATH.exists():
        return f"Error: ERP Excel not found at '{ERP_EXCEL_PATH}'."
    try:
        df = pd.read_excel(str(ERP_EXCEL_PATH))

        # Empty cells read back from Excel become NaN (float64).
        # Cast text columns to str so string assignment via .loc works.
        for col in ['Promise Date', 'Notes']:
            if col in df.columns:
                df[col] = df[col].fillna('').astype(str)

        mask = df['Invoice Number'].astype(str).str.strip() == str(invoice_number).strip()
        if not mask.any():
            return f"Error: Invoice '{invoice_number}' not found in ERP."

        df.loc[mask, 'Promise Date'] = str(promise_date)
        df.loc[mask, 'Status']       = 'Promise Received'
        if notes:
            df.loc[mask, 'Notes'] = str(notes)
        df.to_excel(str(ERP_EXCEL_PATH), index=False)

        ts = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        return (
            f"[{ts}] ERP updated: invoice='{invoice_number}' "
            f"promise_date='{promise_date}' status='Promise Received'."
        )
    except Exception as e:
        return f"Error updating ERP for invoice '{invoice_number}': {str(e)}"


In [99]:
# === LLM AGENTS  (2 runtime agents — written by the Agent Creation Agent during project build) ===


class EmailClassification(BaseModel):
    """Structured output for ParseCustomerResponse agent."""
    email_type:     str = Field(description="'invoice_request' or 'promise_date'")
    invoice_number: str = Field(description="Invoice number e.g. INV-001")
    customer_name:  str = Field(description="Customer name or company name")
    customer_email: str = Field(description="Customer email address")
    promise_date:   str = Field(default="", description="Payment promise date YYYY-MM-DD; empty for invoice_request")


PARSE_SYSTEM_PROMPT = (
    "You are a customer email classifier for an Accounts Receivable dunning system.\n\n"
    "Analyze the customer email and:\n"
    "1. Identify the invoice number (format INV-XXX), customer name, and customer email.\n"
    "2. Classify as:\n"
    "   - 'invoice_request': customer is asking for a copy of their invoice\n"
    "   - 'promise_date': customer is providing a payment commitment date\n"
    "3. Act based on classification:\n"
    "   - invoice_request: call send_invoice_to_customer, then call "
    "update_inquiry_status_database with inquiry_type='invoice_request'\n"
    "   - promise_date: extract date (YYYY-MM-DD), call "
    "update_inquiry_status_database with inquiry_type='promise_date' and the promise_date\n"
)


def create_parse_customer_response_agent():
    """ParseCustomerResponse: classifies emails and returns structured EmailClassification. Model: gemini-3.1-flash-lite"""
    llm   = ChatGoogleGenerativeAI(model=MODELS["parse_agent"], google_api_key=GOOGLE_API_KEY, temperature=0)
    tools = [send_invoice_to_customer, update_inquiry_status_database]
    return create_react_agent(
        llm, tools,
        prompt=PARSE_SYSTEM_PROMPT,
        response_format=EmailClassification,   # guarantees structured output
    )


ERP_SYSTEM_PROMPT = (
    "You are an ERP update agent for an Accounts Receivable system.\n"
    "Update the ERP system with the provided invoice number and promise date "
    "using the update_erp tool. Confirm what was updated."
)


def create_access_update_erp_agent():
    """AccessUpdateERP: updates ERPDirectory/invoices.xlsx with customer promise dates. Model: gemini-3.1-flash-lite"""
    llm   = ChatGoogleGenerativeAI(model=MODELS["erp_agent"], google_api_key=GOOGLE_API_KEY, temperature=0)
    tools = [update_erp]
    return create_react_agent(llm, tools, prompt=ERP_SYSTEM_PROMPT)

In [100]:
PARSE_LOG = OUTPUT_DIR / 'ParseCustomerResponse.log'
ERP_LOG   = OUTPUT_DIR / 'AccessUpdateERP.log'


def _content_to_str(content) -> str:
    """Normalise AIMessage.content to plain string (Gemini can return a list of blocks)."""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict):
                parts.append(item.get('text', str(item)))
            else:
                parts.append(getattr(item, 'text', str(item)))
        return '\n'.join(parts)
    return str(content)


def _get_tokens(msg) -> tuple:
    """Return (input_tokens, output_tokens) for one message.

    usage_metadata is a plain dict in langchain-core 1.x, so use .get() not getattr().
    """
    um = getattr(msg, 'usage_metadata', None)
    if isinstance(um, dict) and um:
        return um.get('input_tokens', 0) or 0, um.get('output_tokens', 0) or 0
    # Fallback: response_metadata (Google GenAI raw format)
    rm = getattr(msg, 'response_metadata', None) or {}
    if isinstance(rm, dict):
        rm_u = rm.get('usage_metadata', {}) or {}
        if isinstance(rm_u, dict) and rm_u:
            return (
                rm_u.get('prompt_token_count', 0) or 0,
                rm_u.get('candidates_token_count', 0) or 0,
            )
    return 0, 0


def _append_agent_log(
    log_path: Path, agent_name: str, email_file: str,
    invoice_number: str, input_text: str, result: dict,
) -> None:
    """Append one timestamped entry — full transcript, per-message tokens, exact LLM output."""
    ts       = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    messages = result.get('messages', [])

    conversation_lines  = []
    total_input_tokens  = 0
    total_output_tokens = 0

    for msg in messages:
        role           = type(msg).__name__
        full_content   = _content_to_str(getattr(msg, 'content', ''))
        msg_in, msg_out = _get_tokens(msg)
        total_input_tokens  += msg_in
        total_output_tokens += msg_out

        # Tool calls on AIMessage
        tool_calls_str = ''
        for tc in getattr(msg, 'tool_calls', []):
            tool_calls_str += f"\n    [ToolCall] {tc['name']}({json.dumps(tc['args'])})"

        # Response metadata (finish_reason etc.) for AI messages
        meta_str = ''
        rm = getattr(msg, 'response_metadata', None)
        if rm and isinstance(rm, dict) and role == 'AIMessage':
            snippet = {k: rm[k] for k in ('finish_reason', 'model_name') if k in rm}
            if snippet:
                meta_str = f"\n    [Meta] {snippet}"

        token_str = f' [tokens in={msg_in} out={msg_out}]' if (msg_in or msg_out) else ''
        conversation_lines.append(
            f'  [{role}]{token_str}\n'
            f'    {full_content}'
            f'{tool_calls_str}'
            f'{meta_str}'
        )

    # Structured response from response_format=EmailClassification
    structured     = result.get('structured_response')
    structured_str = ''
    if structured is not None:
        structured_str = (
            '\nSTRUCTURED RESPONSE:\n'
            f"  email_type    : {getattr(structured, 'email_type', '')}\n"
            f"  invoice_number: {getattr(structured, 'invoice_number', '')}\n"
            f"  customer_name : {getattr(structured, 'customer_name', '')}\n"
            f"  customer_email: {getattr(structured, 'customer_email', '')}\n"
            f"  promise_date  : {getattr(structured, 'promise_date', '')}\n"
        )

    final_output = _content_to_str(messages[-1].content) if messages else '(no output)'

    log_entry = (
        '\n' + '=' * 80 + '\n'
        f'TIMESTAMP      : {ts}\n'
        f'AGENT          : {agent_name}\n'
        f'EMAIL FILE     : {email_file}\n'
        f'INVOICE        : {invoice_number}\n'
        + '-' * 80 + '\n'
        + 'INPUT:\n' + input_text + '\n'
        + '-' * 80 + '\n'
        + 'CONVERSATION TRANSCRIPT (exact LLM output per message):\n'
        + '\n'.join(conversation_lines) + '\n'
        + '-' * 80 + '\n'
        + 'FINAL OUTPUT:\n' + final_output + '\n'
        + structured_str
        + '-' * 80 + '\n'
        + 'TOKEN USAGE:\n'
        + f'  Input tokens  : {total_input_tokens}\n'
        + f'  Output tokens : {total_output_tokens}\n'
        + f'  Total tokens  : {total_input_tokens + total_output_tokens}\n'
        + '=' * 80 + '\n'
    )

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    with open(log_path, 'a', encoding='utf-8') as fh:
        fh.write(log_entry)


def orchestrate() -> None:
    """Main loop: for each email call agents, log every call, move to Processed_Emails/."""
    print('=' * 60)
    print('AR DUNNING MULTI-AGENT ORCHESTRATION')
    print('=' * 60)

    parse_agent = create_parse_customer_response_agent()
    erp_agent   = create_access_update_erp_agent()

    email_files = sorted(INPUT_EMAILS_DIR.glob('*.txt'))
    print(f'\nFound {len(email_files)} email(s) to process.\n')

    for email_file in email_files:
        print(f"\n{'─' * 50}")
        print(f'Processing: {email_file.name}')
        print(f"{'─' * 50}")

        try:
            email_content = email_file.read_text(encoding='utf-8')
            parse_input   = f'Process this customer email:\n\n{email_content}'

            # Step 1: ParseCustomerResponse
            parse_result = parse_agent.invoke({'messages': [('human', parse_input)]})

            structured = parse_result.get('structured_response')
            if structured is not None:
                email_type   = getattr(structured, 'email_type',    'unknown')
                inv_num      = getattr(structured, 'invoice_number','unknown')
                promise_date = getattr(structured, 'promise_date',  '')
                cust_name    = getattr(structured, 'customer_name', '')
            else:
                output_str  = _content_to_str(parse_result['messages'][-1].content)
                parsed_data = {}
                start = output_str.find('{')
                end   = output_str.rfind('}') + 1
                if start >= 0 and end > start:
                    try:
                        parsed_data = json.loads(output_str[start:end])
                    except json.JSONDecodeError:
                        print('  Warning: could not parse JSON from parse agent output.')
                email_type   = parsed_data.get('email_type',    'unknown')
                inv_num      = parsed_data.get('invoice_number','unknown')
                promise_date = parsed_data.get('promise_date',  '')
                cust_name    = parsed_data.get('customer_name', '')

            print(f'  Classified: email_type={email_type!r}  invoice={inv_num!r}')

            _append_agent_log(
                PARSE_LOG, 'ParseCustomerResponse',
                email_file.name, inv_num, parse_input, parse_result,
            )

            # Step 2: AccessUpdateERP (promise_date only)
            if email_type == 'promise_date':
                erp_input = (
                    f'Update ERP for invoice {inv_num} with promise date {promise_date}. '
                    f'Customer: {cust_name}'
                )
                print(f'  -> AccessUpdateERP: {inv_num} promise={promise_date}')
                erp_result = erp_agent.invoke({'messages': [('human', erp_input)]})
                _append_agent_log(
                    ERP_LOG, 'AccessUpdateERP',
                    email_file.name, inv_num, erp_input, erp_result,
                )

        except Exception as e:
            print(f'  ERROR processing {email_file.name}: {e}')
            traceback.print_exc()

        dest = PROCESSED_EMAILS_DIR / email_file.name
        shutil.move(str(email_file), str(dest))
        print(f'  -> Moved {email_file.name} to Processed_Emails/')
        print('  Waiting 15 s before next email...')
        time.sleep(15)

    print(f"\n{'=' * 60}")
    print('All emails processed. Orchestration complete.')
    print(f"{'=' * 60}")

In [101]:
def generate_status_report() -> str:
    """Query invoice_inquiry DB and write an Excel status report to Output/."""
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql_query(
        "SELECT invoice_number, customer_name, customer_email, inquiry_type, "
        "promise_date, status, notes, processed_at "
        "FROM invoice_inquiry ORDER BY processed_at",
        conn,
    )
    conn.close()

    df.columns = [
        "Invoice Number", "Customer Name", "Customer Email",
        "Inquiry Type", "Promise Date", "Status", "Notes", "Processed At",
    ]

    report_path = OUTPUT_DIR / f"StatusReport_{datetime.now().strftime('%Y%m%d_%H%M%S')}.xlsx"

    with pd.ExcelWriter(report_path, engine="openpyxl") as writer:
        df.to_excel(writer, sheet_name="Invoice Inquiry", index=False)

        # Auto-fit column widths
        ws = writer.sheets["Invoice Inquiry"]
        for col in ws.columns:
            max_len = max((len(str(cell.value or "")) for cell in col), default=10)
            ws.column_dimensions[col[0].column_letter].width = min(max_len + 2, 45)

    print(f"Status report saved: {report_path}")
    print(df.to_string(index=False))
    return str(report_path)

In [102]:
def reset_database() -> None:
    """Reset SQLite DB, ERP Excel, and restore processed emails — for re-running tests."""
    print("Resetting system for testing...")

    conn   = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute("DELETE FROM invoice_inquiry")
    try:
        cursor.execute("DELETE FROM sqlite_sequence WHERE name='invoice_inquiry'")
    except sqlite3.OperationalError:
        pass  # sqlite_sequence only exists after the first insert
    conn.commit()
    conn.close()

    try:
        df = pd.read_excel(ERP_EXCEL_PATH)
        df["Promise Date"] = ""
        df["Status"]       = "Overdue"
        df["Notes"]        = ""
        df.to_excel(ERP_EXCEL_PATH, index=False)
        print("ERP Excel reset.")
    except Exception as e:
        print(f"Warning: could not reset ERP Excel: {e}")

    moved = 0
    for f in PROCESSED_EMAILS_DIR.glob("*.txt"):
        shutil.move(str(f), str(INPUT_EMAILS_DIR / f.name))
        moved += 1
    if moved:
        print(f"Moved {moved} email(s) back to Input_Emails/")

    print("Reset complete — system ready for testing.")

## Execution

Run the cell below to process all emails end-to-end.
Call `reset_database()` first to restore a clean state between test runs.

In [103]:
# ── Optional reset before a fresh run ──────────────────────────
reset_database()

# ── Run the AR Dunning pipeline ─────────────────────────────────
orchestrate()
generate_status_report()

Resetting system for testing...
ERP Excel reset.
Reset complete — system ready for testing.
AR DUNNING MULTI-AGENT ORCHESTRATION


C:\Users\USER\AppData\Local\Temp\ipykernel_21092\2461250165.py:32: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  return create_react_agent(
C:\Users\USER\AppData\Local\Temp\ipykernel_21092\2461250165.py:50: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  return create_react_agent(llm, tools, prompt=ERP_SYSTEM_PROMPT)



Found 5 email(s) to process.


──────────────────────────────────────────────────
Processing: email_001.txt
──────────────────────────────────────────────────
  Classified: email_type='invoice_request'  invoice='INV-001'
  -> Moved email_001.txt to Processed_Emails/
  Waiting 15 s before next email...

──────────────────────────────────────────────────
Processing: email_002.txt
──────────────────────────────────────────────────
  Classified: email_type='promise_date'  invoice='INV-002'
  -> AccessUpdateERP: INV-002 promise=2026-06-20
  -> Moved email_002.txt to Processed_Emails/
  Waiting 15 s before next email...

──────────────────────────────────────────────────
Processing: email_003.txt
──────────────────────────────────────────────────
  Classified: email_type='invoice_request'  invoice='INV-003'
  -> Moved email_003.txt to Processed_Emails/
  Waiting 15 s before next email...

──────────────────────────────────────────────────
Processing: email_004.txt
──────────────────────────

'Output\\StatusReport_20260530_182655.xlsx'